# DataCite vs OpenAlex Work Types

Comparison of the work types available in DataCite and OpenAlex and how they are mapped. It includes works from DataCite and OpenAlex with DOIs.
XPac works from OpenAlex have been included where they have a DOI.

| Dataset | Version |
|---------|---------|
| DataCite | 2026-02 Snasphot |
| OpenAlex | 2026-02-03 Snapshot |

## Connect to DuckDB

In [6]:
%reload_ext sql
%sql duckdb:///../data/duckdb/db.db
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = False

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [7]:
%%sql

WITH datacite_total AS (
    SELECT COUNT(DISTINCT doi) AS total_datacite
    FROM datacite.datacite
    WHERE doi IS NOT NULL
),

openalex_total AS (
    SELECT COUNT(*) AS total_openalex
    FROM openalex.openalex_works
),

openalex_xpac AS (
    SELECT COUNT(*) AS openalex_xpac
    FROM openalex.openalex_works
    WHERE is_xpac = TRUE
),

openalex_non_xpac AS (
    SELECT COUNT(*) AS openalex_non_xpac
    FROM openalex.openalex_works
    WHERE is_xpac = FALSE
),

datacite_in_openalex AS (
    SELECT COUNT(DISTINCT d.doi) AS datacite_in_openalex
    FROM datacite.datacite d
    INNER JOIN openalex.openalex_works o ON d.doi = o.doi
    WHERE d.doi IS NOT NULL
),

datacite_in_openalex_xpac AS (
    SELECT COUNT(DISTINCT d.doi) AS datacite_in_openalex_xpac
    FROM datacite.datacite d
    INNER JOIN openalex.openalex_works o ON d.doi = o.doi
    WHERE d.doi IS NOT NULL AND o.is_xpac = TRUE
),

datacite_in_openalex_non_xpac AS (
    SELECT COUNT(DISTINCT d.doi) AS datacite_in_openalex_non_xpac
    FROM datacite.datacite d
    INNER JOIN openalex.openalex_works o ON d.doi = o.doi
    WHERE d.doi IS NOT NULL AND o.is_xpac = FALSE
)

SELECT
    FORMAT('{:,}', dt.total_datacite) AS total_datacite,
    FORMAT('{:,}', dio.datacite_in_openalex) AS datacite_in_openalex,
    FORMAT('{:.2f}%', 100.0 * dio.datacite_in_openalex / dt.total_datacite) AS datacite_in_openalex_pct,

    FORMAT('{:,}', ot.total_openalex) AS total_openalex,
    FORMAT('{:,}', ox.openalex_xpac) AS openalex_xpac,
    FORMAT('{:.2f}%', 100.0 * ox.openalex_xpac / ot.total_openalex) AS openalex_xpac_pct,
    FORMAT('{:,}', onx.openalex_non_xpac) AS openalex_non_xpac,
    FORMAT('{:.2f}%', 100.0 * onx.openalex_non_xpac / ot.total_openalex) AS openalex_non_xpac_pct,

    FORMAT('{:,}', dix.datacite_in_openalex_xpac) AS datacite_in_openalex_xpac,
    FORMAT('{:.2f}%', 100.0 * dix.datacite_in_openalex_xpac / dio.datacite_in_openalex) AS datacite_in_openalex_xpac_pct,

    FORMAT('{:,}', dinx.datacite_in_openalex_non_xpac) AS datacite_in_openalex_non_xpac,
    FORMAT('{:.2f}%', 100.0 * dinx.datacite_in_openalex_non_xpac / dio.datacite_in_openalex) AS datacite_in_openalex_non_xpac_pct
FROM datacite_total dt
CROSS JOIN openalex_total ot
CROSS JOIN openalex_xpac ox
CROSS JOIN openalex_non_xpac onx
CROSS JOIN datacite_in_openalex dio
CROSS JOIN datacite_in_openalex_xpac dix
CROSS JOIN datacite_in_openalex_non_xpac dinx;

Running query in 'duckdb:///../data/duckdb/db.db'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

total_datacite,datacite_in_openalex,datacite_in_openalex_pct,total_openalex,openalex_xpac,openalex_xpac_pct,openalex_non_xpac,openalex_non_xpac_pct,datacite_in_openalex_xpac,datacite_in_openalex_xpac_pct,datacite_in_openalex_non_xpac,datacite_in_openalex_non_xpac_pct
"114,892,160","110,276,031",95.98%,"293,100,851","101,062,751",34.48%,"192,038,100",65.52%,"93,321,643",84.63%,"16,956,357",15.38%


## Datacite: distinct work types, alphabetical

In [8]:
%%sql
    
SELECT DISTINCT work_type 
FROM datacite.datacite 
ORDER BY work_type ASC;

Running query in 'duckdb:///../data/duckdb/db.db'

work_type
Audiovisual
Award
Book
BookChapter
Collection
ComputationalNotebook
ConferencePaper
ConferenceProceeding
DataPaper
Dataset


## DataCite: distinct work types with counts, sorted by count desc

In [9]:
%%sql

SELECT work_type, COUNT(*) AS cnt
FROM datacite.datacite
GROUP BY work_type
ORDER BY cnt DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

work_type,cnt
Dataset,61096286
PhysicalObject,15882298
Text,15837109
Image,9540030
JournalArticle,3011725
Other,1904567
Preprint,1857340
Collection,1203170
Software,823600
None,613264


## OpenAlex: distinct work types, alphabetical

In [10]:
%%sql

SELECT DISTINCT work_type
FROM openalex.openalex_works
ORDER BY work_type ASC;

Running query in 'duckdb:///../data/duckdb/db.db'

work_type
article
book
book-chapter
book-section
database
dataset
dissertation
editorial
erratum
grant


## OpenAlex: distinct work types with counts, sorted by count desc

In [11]:
%%sql

SELECT work_type, COUNT(*) AS cnt
FROM openalex.openalex_works
GROUP BY work_type
ORDER BY cnt DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

work_type,cnt
article,145691737
dataset,63104016
other,36245604
book-chapter,23500254
preprint,6247402
review,3970920
paratext,3826678
book,2707040
dissertation,1699544
letter,1551799


## DataCite to OpenAlex Work Types Mapping

In [12]:
%%sql

SELECT
    d.work_type AS datacite_work_type,
    o.work_type AS openalex_work_type,
    COUNT(*) AS cnt
FROM datacite.datacite d
JOIN openalex.openalex_works o ON d.doi = o.doi
GROUP BY d.work_type, o.work_type
ORDER BY d.work_type ASC, cnt DESC;

Running query in 'duckdb:///../data/duckdb/db.db'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

datacite_work_type,openalex_work_type,cnt
Audiovisual,other,521045
Audiovisual,article,18064
Audiovisual,preprint,560
Audiovisual,dataset,121
Audiovisual,book,63
Audiovisual,dissertation,33
Audiovisual,review,23
Audiovisual,book-chapter,16
Audiovisual,paratext,11
Audiovisual,editorial,2


## DataCite to OpenAlex Work Types Mapping With Examples

In [13]:
%%sql

WITH counts AS (
    SELECT
        d.work_type AS datacite_work_type,
        o.work_type AS openalex_work_type,
        COUNT(*) AS cnt
    FROM datacite.datacite d
    JOIN openalex.openalex_works o ON d.doi = o.doi
    GROUP BY d.work_type, o.work_type
),
ranked_counts AS (
    SELECT
        datacite_work_type,
        openalex_work_type,
        cnt,
        ROW_NUMBER() OVER (
            PARTITION BY datacite_work_type
            ORDER BY cnt DESC
        ) AS rank
    FROM counts
),
examples AS (
    SELECT
        d.work_type AS datacite_work_type,
        o.work_type AS openalex_work_type,
        CONCAT('https://doi.org/', d.doi) AS doi,
        d.title,
        ROW_NUMBER() OVER (
            PARTITION BY d.work_type, o.work_type
            ORDER BY d.doi
        ) AS rn
    FROM datacite.datacite d
    JOIN openalex.openalex_works o ON d.doi = o.doi
    WHERE d.title IS NOT NULL
      AND LENGTH(d.title) > 30
)
SELECT
    rc.datacite_work_type,
    rc.openalex_work_type,
    rc.cnt,
    e.doi,
    e.title
FROM ranked_counts rc
LEFT JOIN examples e
    ON rc.datacite_work_type = e.datacite_work_type
    AND rc.openalex_work_type = e.openalex_work_type
    AND e.rn <= 3
WHERE rc.rank <= 3
ORDER BY rc.datacite_work_type ASC, rc.cnt DESC, e.rn ASC;

Running query in 'duckdb:///../data/duckdb/db.db'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

datacite_work_type,openalex_work_type,cnt,doi,title
Audiovisual,other,521045,https://doi.org/10.11575/prism/41418,Academic integrity policy analysis of Alberta and Manitoba colleges
Audiovisual,other,521045,https://doi.org/10.11575/prism/41685,Developing an OER with an International Collaborative Team
Audiovisual,other,521045,https://doi.org/10.11575/prism/42226,"Academic Integrity Lessons: Practical Ideas for Teaching, Learning, and Assessment"
Audiovisual,article,18064,https://doi.org/10.11581/dtu:00000047,Research Data Management (eLearning course)
Audiovisual,article,18064,https://doi.org/10.11583/dtu.12504533,"University Research Assessment using ORCID, WoS and VIVO"
Audiovisual,article,18064,https://doi.org/10.11583/dtu.12504533.v2,"University Research Assessment using ORCID, WoS and VIVO"
Audiovisual,preprint,560,https://doi.org/10.14288/1.0042748,"Chromatic number, clique subdivisions and the conjectures of Hajos and ErdHos-Fajtlowicz"
Audiovisual,preprint,560,https://doi.org/10.14288/1.0042765,Independent sets in hypergraphs
Audiovisual,preprint,560,https://doi.org/10.14288/1.0042777,Finite u Invariant and Bounds on Cohomology Symbol Lengths
Award,other,8041,https://doi.org/10.17608/k6.auckland.13040615,National eScience Infrastructure Investment Case FINAL 29_10_2010 REDACTED.pdf
